In [0]:
USE CATALOG olist_retail;
USE SCHEMA silver_datasets;
-- CREATE CUSTOMER DIM TABLE
CREATE SCHEMA GOLD_DATASET;

CREATE OR REPLACE TABLE gold_dataset.dim_customer AS
SELECT
    customer_id,
    customer_unique_id,
    customer_city,
    customer_state
FROM customer_silver;

-- CREATE DIM PRODUCT
CREATE OR REPLACE TABLE gold_dataset.dim_product AS
SELECT
    product_id,
    product_category_name,
    product_description_lenght,
    product_photos_qty,
    product_weight_g
FROM products_silver;

-- CREATE DIM SELLERS
CREATE OR REPLACE TABLE gold_dataset.dim_seller AS
SELECT
    seller_id,
    seller_city,
    seller_state
FROM SELLERS_SILVER;

-- CREATE DIM DATE
CREATE OR REPLACE TABLE gold_dataset.dim_date AS
SELECT DISTINCT
    CAST(order_purchase_timestamp AS DATE) AS order_date,
    YEAR(order_purchase_timestamp) AS year,
    MONTH(order_purchase_timestamp) AS month,
    DAY(order_purchase_timestamp) AS day,
    QUARTER(order_purchase_timestamp) AS quarter
FROM order_silver;

-- CREATE FACT SALES
CREATE OR REPLACE TABLE gold_dataset.fact_sales AS
SELECT
    oi.order_id,
    o.customer_id,
    c.customer_unique_id,
    oi.product_id,
    oi.seller_id,
    CAST(o.order_purchase_timestamp AS DATE) AS order_date,
    oi.price,
    oi.freight_value,
    o.order_status,

    p.payment_value,
    p.payment_type,
    p.payment_installments,


    r.review_score,

    DATEDIFF(
        o.order_delivered_customer_date,
        o.order_purchase_timestamp
    ) AS delivery_days

    FROM OITEMS_SILVER oi

    LEFT JOIN ORDER_SILVER o
         ON oi.order_id = o.order_id

    LEFT JOIN CUSTOMER_SILVER C
         ON o.customer_id = c.customer_id

    LEFT JOIN payment_silver p
         ON oi.order_id = p.order_id

    LEFT JOIN review_silver r
         ON oi.order_id = r.order_id;